# Asking the graph

Two engines read the same graph, and they are good at opposite things.

**AQLizer** is Arango's natural-language-to-AQL service. It writes a query, runs it,
and hands back both the rows and the query -- so an answer can be checked rather
than trusted. It is the one to use when the question has a shape: count, sum, rank,
traverse, or *find the ones that are missing something*.

**GraphRAG** is the retriever. It searches the entity descriptions and the source
text, follows the relations it lands on, and writes an answer from what it read.
It is the one to use when the question has no shape -- when it is phrased in words
the model does not use, or when the answer is spread across a dozen files.

Neither is a fallback for the other. The last section asks one question both ways
to show where the line is.

In [1]:
import logging
from sysml import nl

logging.disable(logging.INFO)  # both services narrate every step

---

# Part 1 -- AQLizer

Nothing below is a hand-written query. `sysml/aql_examples.md` teaches the model how
SysML concepts are laid out here -- an `attributes` map, an `owns`/`typedby` tree, a
`stated` flag on the edges -- and the AQL in every answer is what it wrote from that.

## 1. A mass budget

This is the hardest shape in the set: walk up to six hops down the containment tree
through two different edge types, pull two attributes off each element it lands on,
add them, sort by the sum, and cite where each number is declared.

In [2]:
nl.instance().ask(
    "For each Saturn V stage, give its dry mass, its propellant mass and the sum of "
    "the two, sorted by the total, with the file and line each is declared on."
).show(row_limit=7)

Q  For each Saturn V stage, give its dry mass, its propellant mass and the sum of the two, sorted by the total, with the file and line each is declared on.

AQL
   WITH sysml_Entities, sysml_Relations
   FOR e IN sysml_Entities
     FILTER e.entity_name == "SATURNV"
     FOR child, edge IN 1..6 OUTBOUND e sysml_Relations
       FILTER edge.relationship_type IN ["owns", "typedby"]
       FILTER child.entity_type == "part" 
       LET dryMass = child.attributes.dryMass.value
       LET propellantMass = child.attributes.propellantMass.value
       LET totalMass = (dryMass || 0) + (propellantMass || 0)
       FILTER totalMass != 0
       SORT totalMass DESC
       RETURN {
         stage: child.entity_name,
         dryMass: {value: dryMass, unit: "kg"},
         propellantMass: {value: propellantMass, unit: "kg"},
         totalMass: {value: totalMass, unit: "kg"},
         at: CONCAT(child.source_file, ":", child.source_line)
       }

rows (41, first 7)
   {"stage": "S-IC", "dryMass": {

Every figure is a number a file states, and the `file:line` beside it is where. That
is the half of the graph the lexer wrote; an LLM reading the same text reports the
masses as prose and cannot be summed.

## 2. What is *not* there

Coverage questions are the ones a requirements engineer actually asks, and they are
an anti-join: requirements with no incoming `satisfies` edge. Retrieval cannot answer
this at all -- there is no passage describing the absence of a relation.

In [3]:
nl.instance().ask(
    "Which ten Apollo requirements have the most elements satisfying them, and how "
    "many Apollo requirements have none at all?"
).show(row_limit=3)

Q  Which ten Apollo requirements have the most elements satisfying them, and how many Apollo requirements have none at all?

AQL
   WITH sysml_Entities, sysml_Relations
   
   LET satisfiedRequirements = (
     FOR r IN sysml_Relations
       FILTER r.type == "RELATED_TO" AND r.relationship_type == "satisfies"
       FOR e IN sysml_Entities
         FILTER r._to == e._id
         RETURN DISTINCT e._id
   )
   
   LET unsatisfiedRequirements = (
     FOR e IN sysml_Entities
       FILTER e.entity_type == "requirement" AND "apollo-11" IN e.models
       FILTER LENGTH(
         FOR r IN sysml_Relations
           FILTER r._to == e._id AND r.relationship_type == "satisfies"
           RETURN 1
       ) == 0
       RETURN e.entity_name
   )
   
   LET topTenSatisfiedRequirements = (
     FOR e IN sysml_Entities
       FILTER e.entity_type == "requirement" AND "apollo-11" IN e.models
       LET satisfiedCount = LENGTH(
         FOR v, r IN 1..1 INBOUND e._id sysml_Relations
           FILTER

## 3. The graph can be asked how it was built

Every relation carries `stated`: true if the lexer read it out of the syntax, absent
if the LLM inferred it. So "how much of this graph is read and how much is guessed"
is itself a query -- per model, in one pass.

In [4]:
nl.instance().ask(
    "Break the relations down by model and by whether they were read from the "
    "syntax or inferred by the LLM."
).show(row_limit=6)

Q  Break the relations down by model and by whether they were read from the syntax or inferred by the LLM.

AQL
   WITH sysml_Relations, sysml_Entities, sysml_Chunks, sysml_Communities, sysml_Documents
   FOR r IN sysml_Relations
     FILTER r.type == "RELATED_TO"
     LET from = DOCUMENT(r._from)
     FILTER from != null
     FOR m IN from.models
       COLLECT model = m INTO relations
       LET count = LENGTH(relations)
       LET source = FIRST(relations).r.stated == true ? "read from the syntax" : "inferred by the LLM"
       RETURN {
         model: model,
         source: source,
         relations: count
       }
   
   
   The AQL error "variable 'model' is assigned multiple times" was likely caused by using `model` both as a loop variable and as a variable in the `COLLECT` projection. To resolve this, I replaced the loop variable name `model` with `m` to avoid the conflict. The corrected query now should not have any variable assignment conflicts.

rows (3, first 3)
   {"mode

The read majority is the point of the second pass. What the LLM contributes is the
1,000-odd relations that are in the *prose* -- `refines`, `dependson`, `performs` --
which no lexer can find.

## 4. An element has two names, and both work

`requirement def <'HLR-R001'> CrewReturnSafetyRequirement` can be addressed either
way, and an engineer will use the identifier. Extraction, reading prose, kept
whichever name the sentence it was reading happened to use -- so requirements
arrived twice, once under each name, with the edges split between the two halves.
The lexer reads the declaration, which is the only place that says they are one
thing, and folds them together. Asking by the identifier now reaches the whole
element.

In [5]:
nl.instance().ask(
    "Trace the requirement whose short name is HLR-R001: what satisfies it, what "
    "refines it, and what it is related to in either direction."
).show(row_limit=8)

Q  Trace the requirement whose short name is HLR-R001: what satisfies it, what refines it, and what it is related to in either direction.

AQL
   WITH sysml_Entities, sysml_Relations
   LET requirement = FIRST(
     FOR e IN sysml_Entities
       FILTER e.short_name == "HLR-R001"
       RETURN e
   )
   FOR v, r IN 1..1 ANY requirement sysml_Relations
     FILTER r.type == "RELATED_TO"
         AND r.relationship_type IN ["satisfies", "refines", "connects", "exhibits", "owns", "performs", "redefines", "specializes", "subject", "satisfies", "typedby", "transitionsto", "variantof", "valueref"]
     RETURN {
       relation: r.relationship_type,
       related_entity: v.entity_name,
       direction: r._from == requirement._id ? "outgoing" : "incoming"
     }

rows (20, first 8)
   {"relation": "owns", "related_entity": "MISSIONREQUIREMENTSPACKAGE", "direction": "incoming"}
   {"relation": "typedby", "related_entity": "MISSIONREQUIREMENTSPACKAGE::HLR-R001", "direction": "incoming"}
   {"r

## 5. Joining a layer that is not in any file

The `SIMILAR_TO` edges are computed, not declared -- autograph's `SimilarityFinder`
matching entities of the same kind across model boundaries. They are queryable like
anything else, so "what does the drone have in common with Apollo" is a join.

In [6]:
nl.instance().ask(
    "Which requirements does the drone model state that the Apollo model has an "
    "analogous requirement for, and how close are they?"
).show(row_limit=6)

Q  Which requirements does the drone model state that the Apollo model has an analogous requirement for, and how close are they?

AQL
   WITH sysml_Entities, sysml_Relations
   FOR r IN sysml_Relations
     FILTER r.type == 'SIMILAR_TO' 
     LET droneReq = DOCUMENT(r._from)
     LET apolloReq = DOCUMENT(r._to)
     FILTER droneReq != null 
         AND apolloReq != null
         AND 'drone-logical' IN droneReq.models 
         AND droneReq.entity_type == 'requirement'
         AND 'apollo-11' IN apolloReq.models 
         AND apolloReq.entity_type == 'requirement'
     RETURN {
       droneRequirement: droneReq.entity_name, 
       apolloRequirement: apolloReq.entity_name,
       cosine: r.cosine
     }

rows (10, first 6)
   {"droneRequirement": "DRONEENGINESTAKEHOLDERREQUIREMENTS", "apolloRequirement": "STAKEHOLDERNEED", "cosine": 0.6067132499531839}
   {"droneRequirement": "DRONEENGINESTANDARDSTAKEHOLDERREQUIREMENTS", "apolloRequirement": "STAKEHOLDERNEED", "cosine": 0.605806908873

### Where AQLizer stops

It needs the question to land on a field. Ask it something whose answer is spread
through the `doc` comments of a dozen requirements in four files and there is no
column to filter on -- which is the next section.

---

# Part 2 -- GraphRAG

Three scopes, all upstream, all reading this graph.

  `local`    hybrid vector + BM25 over the entities, fused, then expanded over the
             relations it lands on
  `unified`  the source chunks and the entity graph searched in parallel
  `global`   the community reports, map-reduced

## 6. `local` -- a question in words the model never uses

No SysML file contains "alive", "breathing" or "keeps". The elements are called
`PLSS`, `PSA`, `EnvironmentalControlSystem`. Vector search does not care.

In [7]:
(await nl.retriever().ask_async(
    "What keeps the astronauts alive and breathing, and what limits does it "
    "have to hold?"
)).show(row_limit=4)

Q  What keeps the astronauts alive and breathing, and what limits does it have to hold?

retrieved  25 documents, 42 edges, 58,419 chars of context

cited (13, first 4)
   {"cite": 1, "source": "models/apollo-11-sysml-v2/Requirements/MissionRequirementsPackage.sysml"}
   {"cite": 10, "source": "models/apollo-11-sysml-v2/Execution/Apollo11MissionExecutionPackage.sysml"}
   {"cite": 11, "source": "models/apollo-11-sysml-v2/Technical/TechnicalIndividualsPackage.sysml"}
   {"cite": 12, "source": "models/apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml"}

A  ## Answer

The astronauts' life support and breathing are maintained by the Portable Life Support System (PLSS), which provides essential systems such as oxygen supply and contaminant removal. Specifically, the requirement CLR-R024 states that the PLSS shall contain and supply breathable oxygen to the crew member, which is crucial for extravehicular activities [CITE:7]. Additionally, the PLSS must remove carbon dioxide

## 7. `unified` -- a figure that never became an entity

Some numbers live only in a `doc` comment, so they are in the source text and in no
`attributes` map. `unified` searches the chunks and the graph together, which is what
reaches them.

The second call is the important one: `evidence(find=...)` prints the retrieved text
around the figure, so the answer can be checked against what was actually read rather
than taken on trust.

In [8]:
answer = await nl.retriever().ask_async(
    "How much drinking water must the environmental control system supply "
    "per crew member per day?",
    scope="unified")
answer.show(row_limit=3)
answer.evidence(chars=420, find="water")

Could not initialize local retriever: The token is not yet valid (iat)


No chunks available - using entities only for LLM input


Q  How much drinking water must the environmental control system supply per crew member per day?

retrieved  0 documents

A  I couldn't find any relevant information for your query: 'How much drinking water must the environmental control system supply per crew member per day?'.

evidence  (none retrieved)


The number in the answer is in the `doc` comment printed underneath it, and the
citation resolves to the file it came from. That is the difference between a
retrieval and a recollection.

In [9]:
answer = await nl.retriever().ask_async(
    "What is the minimum delta-v the lunar module ascent stage has to provide, "
    "and why that figure?",
    scope="unified")
answer.show(row_limit=3)
answer.evidence(chars=320, find="delta")

No chunks available - using entities only for LLM input


Q  What is the minimum delta-v the lunar module ascent stage has to provide, and why that figure?

retrieved  0 documents

A  I couldn't find any relevant information for your query: 'What is the minimum delta-v the lunar module ascent stage has to provide, and why that figure?'.

evidence  (none retrieved)


## 8. `global` -- a question no single row answers

`global` never looks at an entity. It reads the 138 community reports the extraction
step wrote, scores them against the question, and summarises the ones that survive --
so it answers about the corpus as a whole.

In [10]:
(await nl.retriever().ask_async(
    "What concerns are these models organised around, and what does each part "
    "of the corpus contribute?",
    scope="global")).show()

Q  What concerns are these models organised around, and what does each part of the corpus contribute?

retrieved  22 community reports -> 22 points

A  # Overview of Concerns in the Apollo 11 SysML v2 Models

The provided set of SysML v2 models for Apollo 11 is structured around several key concerns central to the mission's success. Each part of the corpus contributes specific elements related to these major themes:

## Command Module and Mission Requirements

- **Apollo 11 Mission System and Command Module Community**: This part focuses on ensuring the command module meets mission-critical requirements such as pressure maintenance, safety, and reentry operations. This aspect underscores its vital role in mission success and crew safety.

- **Apollo 11 Mission System Command Module and Requirements Community**: It addresses how the command module fulfills structural and functional requirements, documented through specific requirements like CLR-R075 and CLR-R076, ensuring operational ca

---

# The line between them

One question, both engines.

In [11]:
QUESTION = "How many Apollo requirements does nothing satisfy?"

nl.instance().ask(QUESTION).show(row_limit=2)

Q  How many Apollo requirements does nothing satisfy?

AQL
   WITH sysml_Entities, sysml_Relations
   LET unsatisfied = (
     FOR e IN sysml_Entities
       FILTER e.entity_type == 'requirement' AND 'apollo-11' IN e.models
       FILTER LENGTH(FOR r IN sysml_Relations
                FILTER r._to == e._id AND r.relationship_type == 'satisfies'
                RETURN 1) == 0
       RETURN e.entity_name)
   RETURN {total: LENGTH(unsatisfied), examples: SLICE(unsatisfied, 0, 10)}

rows (1, first 1)
   {"total": 371, "examples": ["REQUIREMENT CLR-R069", "FUNCTIONALREQUIREMENTSPACKAGE: FLR-R066", "CREW EMERGENCY MEDICAL PROFICIENCY", "LOW EARTH ORBIT DELTA V", "COMMUNICATIONS", "APOLLO11PHASES.TEC", "REQUIREMENT CLR-R068", "APOLLO11PHASES.REENTRYLANDING", "REQUIREMENTDEFINITION", "SHN-N029"]}

A  Among the Apollo requirements, 371 are currently not satisfied by any entities. Some examples of these unsatisfied requirements include 'REQUIREMENT CLR-R069', 'FUNCTIONALREQUIREMENTSPACKAGE: FLR-

In [12]:
(await nl.retriever().ask_async(QUESTION)).show(row_limit=3)

Q  How many Apollo requirements does nothing satisfy?

retrieved  23 documents, 40 edges, 39,811 chars of context

cited (8, first 3)
   {"cite": 1, "source": "models/apollo-11-sysml-v2/Purpose/MissionSpecificationPackage.sysml"}
   {"cite": 2, "source": "models/apollo-11-sysml-v2/CoSMA/CoSMAViewsPackage.sysml"}
   {"cite": 3, "source": "models/apollo-11-sysml-v2/Analysis/AnalysisPackage.sysml"}

A  ## Analysis of Apollo Requirements Satisfaction

To determine how many Apollo requirements are not satisfied by any action or operation, we extract information from the provided SysML v2 model context.

### List of Unsatisfied Requirements

In the retrieved context, the model provides a comprehensive list of requirements and the operations that satisfy them. The process for identifying unsatisfied requirements is straightforward: any requirement that is not explicitly associated with any satisfying operation is considered unsatisfied.

### Results

Upon reviewing the provided sources, a var

AQLizer counts them. GraphRAG cannot: it retrieves requirements that *look* relevant
and describes them, because there is no passage anywhere that states how many
requirements lack a relation. Reverse the question -- "what keeps the astronauts
alive" -- and AQLizer has nothing to filter on while GraphRAG answers from four files.

So the rule is about the question, not the engine:

| the question is about | use |
|---|---|
| a number, a count, a ranking, a rollup | AQLizer |
| something absent -- unsatisfied, unowned, uncovered | AQLizer |
| provenance, or the shape of the graph itself | AQLizer |
| a concept the model spells differently | GraphRAG `local` |
| a figure written in prose rather than declared | GraphRAG `unified` |
| the corpus as a whole | GraphRAG `global` |

Both are pointed at a graph the importer's own writer produced, and neither has a
hand-written query behind it. When an answer is wrong, the fix goes in
`sysml/aql_examples.md` -- two of the queries above are only correct because a
previous wrong answer was turned into a worked example there.